# 20 — Screener Pipeline (Combined Workflow)

Composes `data`, `indicators`, `selection`, and `risk` modules end-to-end. If you ran 10, 11, and 12, this is those APIs working together.

In [ ]:
from __future__ import annotations
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

from swing_screener.data import load_universe_from_package, fetch_ohlcv, MarketDataConfig
from swing_screener.indicators.trend import TrendConfig, compute_trend_features
from swing_screener.indicators.momentum import MomentumConfig, compute_momentum_features
from swing_screener.indicators.volatility import VolatilityConfig, compute_volatility_features
from swing_screener.selection.ranking import RankingConfig, compute_hot_score
from swing_screener.selection.universe import UniverseFilterConfig, apply_universe_filters
from swing_screener.selection.entries import EntrySignalConfig, build_signal_board
from swing_screener.risk import RiskConfig, build_trade_plans

In [ ]:
# Load first 20 tickers from a broad-market universe and fetch daily OHLCV
tickers = load_universe_from_package("broad_market_stocks")[:20]
print(f"Universe: {len(tickers)} tickers")

ohlcv = fetch_ohlcv(
    tickers + ["SPY"],
    cfg=MarketDataConfig(start="2023-06-01", end="2024-12-31"),
)
print(f"OHLCV shape: {ohlcv.shape}")
print(f"Date range: {ohlcv.index[0].date()} to {ohlcv.index[-1].date()}")
ohlcv.iloc[:3, :6]

In [ ]:
# Compute trend, momentum, and volatility features independently, then join
trend = compute_trend_features(ohlcv, TrendConfig())
momentum = compute_momentum_features(ohlcv, MomentumConfig())
volatility = compute_volatility_features(ohlcv, VolatilityConfig(atr_window=14))

# Join all features and add currency (all tickers are USD-denominated)
features = trend.join(momentum, how="inner").join(volatility, how="inner")
features["currency"] = "USD"
print(f"Features shape: {features.shape}")
features.head()

In [ ]:
# Apply universe filters (price, ATR, trend, currency) and rank by momentum/RS
filter_cfg = UniverseFilterConfig(min_price=5, max_price=1000, min_avg_daily_volume_eur=0)
filtered = apply_universe_filters(features, cfg=filter_cfg)

eligible = filtered[filtered["is_eligible"]].copy()
print(f"Eligible: {len(eligible)} / {len(filtered)}")

ranked = compute_hot_score(eligible, RankingConfig())
ranked[["last", "atr14", "mom_6m", "mom_12m", "rs_6m", "score", "rank"]].head(10)

In [ ]:
# Build signal board from OHLCV data, then create trade plans with R-based sizing
signal_board = build_signal_board(
    ohlcv,
    tickers,
    cfg=EntrySignalConfig(min_history=126),
)
print(f"Signal board entries: {len(signal_board)}")
print(f"Signals found: {(signal_board['signal'] != 'none').sum()}")

risk_cfg = RiskConfig(
    account_size=50000,
    account_currency="USD",
    risk_pct=0.01,
    k_atr=2.0,
    min_rr=2.0,
)
plans = build_trade_plans(ranked, signal_board, cfg=risk_cfg)
print(f"Trade plans: {len(plans)}")

if not plans.empty:
    plans[["signal", "entry", "stop", "shares", "position_value", "realized_risk"]].head(10)

In [ ]:
# Final output table: ranked candidates with position sizes
output = ranked.head(10).copy()
if plans is not None and not plans.empty:
    output = output.merge(plans, left_index=True, right_index=True, how="left")
cols = [c for c in ["last", "sma200", "atr14", "score", "shares", "position_value", "realized_risk"] if c in output.columns]
output[cols].head(10)